In [3]:
import os
import warnings
import MDAnalysis as mda
import pandas as pd
import prolif as plf

warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis")

homedir = os.getenv("HOME")
datadir = os.path.join(homedir, "mnt/gdrive/data/GROMACS/Cuedc2-ligands/")

# CUEDC2-Rifamycin

## Docked

In [4]:


complex_dir = os.path.join(datadir, "cuedc2_rif")
complex_pdb = os.path.join(complex_dir, "complex/complex_box.pdb")

# 1. Load PDB & build topology
u = mda.Universe(complex_pdb)
u.guess_TopologyAttrs(to_guess=["elements"])
u.atoms.guess_bonds()

# 2. Select protein and ligand
protein_atoms = u.select_atoms("protein")
ligand_atoms = u.select_atoms("resname RIF")

# 3. Create ProLIF molecules
protein_mol = plf.Molecule.from_mda(protein_atoms)
ligand_mol = plf.Molecule.from_mda(ligand_atoms)

# 4. Run Fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol, n_jobs=1)

# 5. Display 2D Diagram
view = fp.plot_lignetwork(ligand_mol, frame=0)
view

100%|██████████| 1/1 [00:00<00:00,  6.97it/s]


## Equilibrated

In [ ]:
complex_pdb = os.path.join(complex_dir, "prod/start.pdb")

# 1. Load PDB & build topology
u = mda.Universe(complex_pdb)
u.guess_TopologyAttrs(force_guess=["elements", "types"])
u.atoms.guess_bonds()
# 2. Select protein and ligand
protein_atoms = u.select_atoms("protein")
ligand_atoms = u.select_atoms("not protein and not resname HOH and not resname TIP3")

# 3. Create ProLIF molecules
protein_mol = plf.Molecule.from_mda(protein_atoms)
ligand_mol = plf.Molecule.from_mda(ligand_atoms)

# 4. Run Fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol, n_jobs=1)

# 5. Display 2D Diagram
view = fp.plot_lignetwork(ligand_mol, frame=0)
view

100%|██████████| 1/1 [00:00<00:00,  4.93it/s]


## Production End

In [ ]:
complex_pdb = os.path.join(complex_dir, "prod/end.pdb")

# 1. Load PDB & build topology
u = mda.Universe(complex_pdb)
u.guess_TopologyAttrs(force_guess=["elements", "types"])
u.atoms.guess_bonds()
# 2. Select protein and ligand
protein_atoms = u.select_atoms("protein")
ligand_atoms = u.select_atoms("not protein and not resname HOH and not resname TIP3")

# 3. Create ProLIF molecules
protein_mol = plf.Molecule.from_mda(protein_atoms)
ligand_mol = plf.Molecule.from_mda(ligand_atoms)

# 4. Run Fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol, n_jobs=1)

# 5. Display 2D Diagram
view = fp.plot_lignetwork(ligand_mol, frame=0)
view

100%|██████████| 1/1 [00:00<00:00,  4.37it/s]


# cuedc2-Ergotamine


### Docked

In [6]:
complex_dir = os.path.join(datadir, "cuedc2_erg")
complex_pdb = os.path.join(complex_dir, "complex/complex_box.pdb")

# 1. Load PDB & build topology
u = mda.Universe(complex_pdb)
u.guess_TopologyAttrs(force_guess=["elements", "types"])
u.atoms.guess_bonds()
# 2. Select protein and ligand
protein_atoms = u.select_atoms("protein")
ligand_atoms = u.select_atoms("not protein and not resname HOH and not resname TIP3")

# 3. Create ProLIF molecules
protein_mol = plf.Molecule.from_mda(protein_atoms)
# Imperfect protonation, have to add implicit hydrogens
ligand_mol = plf.Molecule.from_mda(ligand_atoms, implicit_hydrogens=True)

# 4. Run Fingerprint
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol, n_jobs=1)

# 5. Display 2D Diagram
view = fp.plot_lignetwork(ligand_mol, frame=0)
view

100%|██████████| 1/1 [00:00<00:00, 12.27it/s]


### equilibrated

> Note: ERG was imperfectly protonated by openababel, so a reference structure is needed to get bond information. Default settings yielded "can't kekulize" errors.


If the ligand was imperfectly protonated (by openbabel), how badly will that affect the MD simulation?


Ran command: `cat /home/daneel/mnt/gdrive/data/GROMACS/Cuedc2-ligands/cuedc2_erg/02_run_acpype.sh
head -n 40 /home/daneel/mnt/gdrive/data/GROMACS/Cuedc2-ligands/cuedc2_erg/ligand/ERG.acpype/acpype.log`
Ran command: `head -n 25 /home/daneel/mnt/gdrive/data/GROMACS/Cuedc2-ligands/cuedc2_erg/ligand/ERG.acpype/ERG_GMX.gro
wc -l /home/daneel/mnt/gdrive/data/GROMACS/Cuedc2-ligands/cuedc2_erg/ligand/ERG.acpype/ERG_GMX.gro`
Ran command: `head -n 60 /home/daneel/mnt/gdrive/data/GROMACS/Cuedc2-ligands/cuedc2_erg/ligand/ERG.acpype/ERG_GMX.itp`
Ran command: `python3 -c '
import pandas as pd
import numpy as np

# Let us check rmsd.csv if present
import os
if os.path.exists("rmsd.csv"):
    df = pd.read_csv("rmsd.csv")
    print(df.head())
    print("Columns:", df.columns)
'`
Ran command: `/usr/local/miniforge3/envs/prolif/bin/python -c '
import pandas as pd
import os
if os.path.exists("rmsd.csv"):
    df = pd.read_csv("rmsd.csv")
    print(df.head())
    print("Columns:", df.columns)
'`

The impact is **substantial** for quantitative conclusions, but depends on the questions you are trying to answer with the simulation. 

Here is a breakdown of why this happens, how it affects the physics of the simulation, and what it means for your results.

---

### 1. Root Cause: The "Docking vs. MD" Mismatch

In `01_extract_and_prepare_ligand.py`, OpenBabel was run with `--AddPolarH`:
```bash
obabel -ipdb raw.pdb -opdb -O prot.pdb --AddPolarH
```
* **For Docking (e.g., AutoDock Vina)**: `--AddPolarH` is standard because scoring functions use a **united-atom** model (carbon atoms implicitly absorb non-polar hydrogens).
* **For MD Simulations (GROMACS + AMBER/GAFF)**: GAFF is strictly an **all-atom** force field. Ergotamine has 33 carbons and should have **73 atoms** total, but only **46 atoms** (43 heavy atoms + 3 polar hydrogens) were supplied to ACPYPE and simulated.

---

### 2. How This Affects the MD Simulation Physics

#### A. Severe: Deflated van der Waals Volume & Pocket Packing
In an all-atom force field (AMBER/GAFF2), carbon Lennard-Jones parameters ($\sigma \approx 3.4\text{ \AA}$) assume explicit hydrogen spheres ($\sigma \approx 2.5\text{ \AA}$) extend outward from the carbon skeleton.
* Because GAFF does **not** enlarge carbon radii to compensate for missing hydrogens (unlike true united-atom force fields such as GROMOS), the ligand in your simulation was **unnaturally shrunken** in volume.
* **Effect**: Water molecules and protein side chains could pack unnaturally close to the ligand core. Binding pocket residues may contract inward or adopt distorted geometries to close the artificial void space.

#### B. Severe: Distorted AM1-BCC Partial Charges
ACPYPE uses Antechamber and SQM to calculate semi-empirical quantum mechanical partial charges (`AM1-BCC`).
* The QM calculation was forced to solve electron density for a molecule missing 27 non-polar protons and their corresponding electrons.
* Antechamber logged a warning: 
  `Warning: The assigned bond types may be wrong... adjust atom valence penalty parameters`.
* **Effect**: Partial charges on the carbon atoms were shifted to compensate for the missing CH/CH₂ dipoles, altering the electrostatic landscape of the ligand.

#### C. Moderate: Torsional Barriers and Flexibility
Rotational energy barriers around single bonds (e.g., the benzyl group rotation, proline puckering) rely heavily on 1–4 steric clashes between hydrogens (H–C–C–H and H–C–C–C dihedrals).
* Without explicit hydrogens, many of these dihedral terms vanished from the GROMACS topology (`ERG_GMX.itp`).
* **Effect**: The ligand likely had **unrealistically low rotational barriers**, making it floppy or prone to visiting conformations that are sterically hindered in reality.

#### D. Minor / Acceptable: Net Ionization State
Ergotamine has a tertiary amine on the ergoline core ($pK_a \approx 6.3 - 6.8$). 
* At physiological pH ($\approx 7.4$), around 75–85% of ergotamine is in the neutral (deprotonated) state.
* Running with net charge `-n 0` in ACPYPE was chemically reasonable for the dominant neutral species, so the net charge itself is not the primary issue.

---

### 3. Summary: Can You Use the Current Simulation?

| Goal / Analysis | Reliability | Verdict |
| :--- | :--- | :--- |
| **Gross Pocket Localization** (Did the ligand leave the pocket?) | Moderate | **Usable as a preliminary sanity check.** The ligand's heavy-atom scaffold still occupied the binding region. |
| **Qualitative Contact Map** (Which residues are in the neighborhood?) | Moderate | **Partially usable.** Residues identified (TRP166, GLN163, GLU194, ASN197) are likely true pocket residues. |
| **Specific Interaction Types** ($\pi$-$\pi$ stacking, exact H-bonds) | Low | **Unreliable.** Aromatic rings and van der Waals contact distances were artificially contracted. |
| **Binding Free Energy (MM-PBSA / MM-GBSA)** | Very Low | **Invalid.** Missing 27 atoms will yield completely erroneous solvation energies and Lennard-Jones terms. |
| **Publication / Formal Presentation** | None | **Must be re-run.** Any reviewer or computational chemist will immediately flag a GAFF ligand simulated with missing non-polar hydrogens. |

---

### 4. How to Fix for a Proper Re-run

To run a fully validated all-atom simulation, update Step 4 in `01_extract_and_prepare_ligand.py`:

```python
# Replace --AddPolarH with -h (adds ALL hydrogens)
ret = subprocess.run(
    [
        "obabel",
        "-ipdb", raw_pdb,
        "-opdb", "-O", prot_pdb,
        "-h",  # Adds all explicit hydrogens (both polar and non-polar)
    ],
    capture_output=True, text=True
)
```

With `-h`, OpenBabel produces the full ~73-atom ergotamine molecule. ACPYPE will then parametrize proper GAFF2 atom types, standard AM1-BCC charges, and complete dihedral potentials without valence errors.

In [9]:
from rdkit import Chem

# 1. Load PDB & build topology for protein only
u = mda.Universe(complex_pdb)
u.guess_TopologyAttrs(force_guess=["elements", "types"])

protein_atoms = u.select_atoms("protein")
protein_atoms.guess_bonds()  # Only guess bonds for protein!
protein_mol = plf.Molecule.from_mda(protein_atoms)

# 2. Select ligand atoms from trajectory
ligand_atoms = u.select_atoms("not protein and not resname HOH and not resname TIP3")

# 3. Load reference topology from mol2 and sanitize
mol2_path = os.path.join(complex_dir, "ligand/ERG.acpype/ERG.mol2")
ref_mol = Chem.MolFromMol2File(mol2_path, sanitize=False)
ref_mol.GetAtomWithIdx(30).SetNumExplicitHs(1)  # indole NH
Chem.SanitizeMol(ref_mol)

# 4. Copy 3D coordinates from the simulation snapshot into the reference conformer
conf = ref_mol.GetConformer()
for i in range(len(ligand_atoms)):
    conf.SetAtomPosition(i, ligand_atoms[i].position.tolist())

# 5. Create ProLIF molecule
ligand_mol = plf.Molecule.from_rdkit(ref_mol, resname="ERG")

# 6. Run Fingerprint & Display 2D Diagram
fp = plf.Fingerprint()
fp.run_from_iterable([ligand_mol], protein_mol, n_jobs=1)

view = fp.plot_lignetwork(ligand_mol, frame=0)
view

100%|██████████| 1/1 [00:00<00:00, 15.73it/s]
